# Feature Type Selection in Machine Learning

Before training a model, features must be separated into meaningful groups. Numerical and categorical features are handled differently, so misclassifying them can distort preprocessing, training, and interpretation.

This notebook shows how to identify, separate, validate, and preprocess these feature types in a disciplined way.

## Why feature type selection matters

Feature type selection affects:

- Encoding strategy
- Scaling decisions
- Model compatibility
- Interpretability
- Training stability

A column stored as integers is not automatically numerical in meaning. A column with text labels is not automatically unusable. The goal is to decide how the model should interpret each feature.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

rng = np.random.default_rng(7)
n_rows = 600

df = pd.DataFrame({
    'CustomerID': [f'C{i:04d}' for i in range(n_rows)],
    'tenure': rng.integers(1, 72, size=n_rows),
    'MonthlyCharges': rng.normal(70, 15, size=n_rows).round(2),
    'TotalCharges': rng.normal(2500, 800, size=n_rows).round(2),
    'gender': rng.choice(['Male', 'Female'], size=n_rows),
    'SeniorCitizen': rng.integers(0, 2, size=n_rows),
    'Partner': rng.choice(['Yes', 'No'], size=n_rows),
    'Dependents': rng.choice(['Yes', 'No'], size=n_rows),
    'Contract': rng.choice(['Month-to-month', 'One year', 'Two year'], size=n_rows),
    'PaymentMethod': rng.choice(['Credit Card', 'Bank Transfer', 'Electronic Check'], size=n_rows),
    'InternetService': rng.choice(['DSL', 'Fiber optic', 'No'], size=n_rows),
})

logit = (
    -0.03 * df['tenure']
    + 0.015 * df['MonthlyCharges']
    + 0.0005 * df['TotalCharges']
)
prob = 1 / (1 + np.exp(-logit))
df['Churn'] = (rng.random(n_rows) < prob).astype(int)

df.head()

In [ ]:
TARGET_COLUMN = 'Churn'
NUMERICAL_FEATURES = ['tenure', 'MonthlyCharges', 'TotalCharges']
CATEGORICAL_FEATURES = ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'Contract', 'PaymentMethod', 'InternetService']
EXCLUDED_COLUMNS = ['CustomerID']

X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

assert TARGET_COLUMN not in NUMERICAL_FEATURES + CATEGORICAL_FEATURES
assert not set(NUMERICAL_FEATURES).intersection(CATEGORICAL_FEATURES)
assert not set(EXCLUDED_COLUMNS).intersection(NUMERICAL_FEATURES + CATEGORICAL_FEATURES)

print({'numerical_features': NUMERICAL_FEATURES, 'categorical_features': CATEGORICAL_FEATURES, 'excluded_columns': EXCLUDED_COLUMNS})

In [ ]:
X = X.drop(columns=EXCLUDED_COLUMNS)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore')),
])

preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, NUMERICAL_FEATURES),
    ('categorical', categorical_pipeline, CATEGORICAL_FEATURES),
])

model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000)),
])

model.fit(X_train, y_train)
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print({'test_accuracy': round(accuracy, 3), 'train_rows': len(X_train), 'test_rows': len(X_test)})

## Practical rules

- Numerical features represent magnitude and usually benefit from scaling and numerical imputation.
- Categorical features represent labels and usually need encoding before modeling.
- Binary columns require judgment: they may be stored as integers but still behave like categories.
- IDs, raw timestamps, target columns, and post-outcome columns should be excluded from feature groups.
- Always split the dataset before fitting preprocessing, and keep the feature lists explicit and documented.

## Validation checklist

- Target is removed before feature grouping.
- Numerical and categorical feature lists do not overlap.
- Excluded columns are not used for training.
- Feature meaning matches preprocessing choice.
- Pipeline is fit only on training data.

## Bonus resources

- Scikit-learn ColumnTransformer Guide
- Google ML Guide - Data Preparation